# Memory Diagrams Demo — Session 03: Mapping Copies in Memory

This notebook draws the memory pictures that make copying intuitive. Instead of memorising definitions, you will see the boxes, pointers, and shared objects that a shallow or deep copy actually produces, and then meet three real-world wrinkles: nested dictionaries, tuples that contain lists, and the JSON round-trip shortcut.

**The big picture.** Memory is a warehouse of objects, and variables are labels pointing at shelves. Copying is a question of *which shelves are rebuilt* and *which goods remain shared*. A diagram turns that question into something you can literally see.

**Why it matters.** Nested dictionaries are everywhere — configuration files, employee records, API payloads. Knowing whether two records share a nested list, and how to get a truly independent clone, prevents silent corruption and makes debugging predictable.

**Roadmap.** In this notebook you will:
1. Read step-by-step ASCII diagrams of shallow and deep copies.
2. Apply the shallow trap to a nested employee registry.
3. Fix it with `copy.deepcopy`.
4. Examine the tuple-of-lists edge case.
5. Use `json.loads(json.dumps(...))` as a quick deep copy and discover its traps.
6. Follow a decision flowchart for choosing shallow or deep.

**Key definitions.** *Nested structure*: a container whose elements are themselves containers. *Reference graph*: the network of objects linked by pointers. *JSON round-trip*: serialising to text and parsing back, which deep-copies only JSON-compatible data. *Type coercion*: silent changes such as tuples becoming lists.

## 1. Memory Diagrams — Shallow vs Deep

A diagram makes the difference obvious. Below, `ptr1` and `ptr2` are references from an outer list to its inner lists.

```text
ORIGINAL:
matrix --> [ outer ] --ptr1--> [1,2,3]     ptr2--> [4,5,6]

SHALLOW = matrix.copy():
matrix --> [ outer ] --ptr1--> [1,2,3] <--ptr1'-- [ outer ] <-- shallow
           (new outer) ptr2--> [4,5,6] <--ptr2'-- (new)
                              ^
                     INNER objects shared!

DEEP = copy.deepcopy(matrix):
matrix --> [ outer ] -ptr1-> [1,2,3]      ptr2-> [4,5,6]
deep   --> [ outer ] -ptr1-> [1,2,3]'     ptr2-> [4,5,6]'   # inner also new
```

Read it as a rule: a shallow copy rebuilds the shelf labels but reuses the goods; a deep copy rebuilds both. Everything that follows is a variation on this picture.

## 2. The Nested Dictionary Twist

Dictionaries make the sharing even easier to miss, because the nested mutable is hidden one level down. With `registry = {1001: {'name': ..., 'skills': [...]}}`, a `registry.copy()` gives a new outer dictionary, but `registry[1001]` — the inner dictionary — is shared, and so is its `skills` list.

Appending to `shallow_reg[1001]['skills']` therefore changes the original registry too. The outer ids differ, which is why the bug passes a casual check, but the inner ids match.

This is the classic trap in record processing: you copy a registry to avoid disturbing the source, edit a nested field, and discover the source moved with you. The fix is `copy.deepcopy`, which rebuilds the nested dictionary and list as well.

In [1]:
# 🐔 NESTED DICT twist — employee registry ka aaina-darr! 🗂️
import copy

emp_1001 = {"name": "Sai Das", "skills": ["python", "sql"], "dept": "Clothing"}
registry = {1001: emp_1001}

shallow_reg = registry.copy()                          # outer dict fresh...
print("outer different:", id(registry) != id(shallow_reg), "✅")
print("skills shared? :", id(registry[1001]["skills"]) == id(shallow_reg[1001]["skills"]), "😱")

# Shallow me inner list mutate 😱 — DONO registries me dikhta!
shallow_reg[1001]["skills"].append("excel")
print("\nshallow_reg :", shallow_reg[1001]["skills"])
print("registry    :", registry[1001]["skills"], "← OG me bhi aa gaya! 📎")
# Expected OUTPUT:
# outer different: True ✅
# skills shared? : True 😱
#
# shallow_reg : ['python', 'sql', 'excel']
# registry    : ['python', 'sql', 'excel'] ← OG me bhi aa gaya! 📎

outer different: True ✅
skills shared? : True 😱

shallow_reg : ['python', 'sql', 'excel']
registry    : ['python', 'sql', 'excel'] ← OG me bhi aa gaya! 📎


## 3. Deep Copy for Registries

`copy.deepcopy(reg2)` produces a registry that shares nothing with the original: a new outer dictionary, a new inner dictionary, and a new `skills` list. Appending to the copy's skills leaves `reg2` exactly as it was, proven by comparing the inner list `id()`s.

The cost is proportional to the size of the structure, so deep copy is the right tool in two situations: when nested mutables may change, and when you need a guaranteed snapshot. For flat data, or for structures you only read, a shallow copy is cheaper and sufficient.

In [2]:
# 🐔 deepcopy — registry poorn-safe! ✂️🗂️
emp_2002 = {"name": "Kavya Gupta", "skills": ["java", "aws"]}
reg2 = {2002: emp_2002}
deep_reg = copy.deepcopy(reg2)
deep_reg[2002]["skills"].append("docker")
print("deep_reg:", deep_reg[2002]["skills"])
print("reg2    :", reg2[2002]["skills"], "← untouched! registry-divorce 💔✅")
print("skills ids differ:", id(reg2[2002]["skills"]) != id(deep_reg[2002]["skills"]))
# Expected OUTPUT:
# deep_reg: ['java', 'aws', 'docker']
# reg2    : ['java', 'aws'] ← untouched! registry-divorce 💔✅
# skills ids differ: True

deep_reg: ['java', 'aws', 'docker']
reg2    : ['java', 'aws'] ← untouched! registry-divorce 💔✅
skills ids differ: True


## 4. The Tuple-of-Lists Edge Case

Tuples are immutable, so Python is free to reuse them rather than copy. Slicing a tuple (`mixed[:]`) returns the **same object** — `s is mixed` is `True` — because there is no way to mutate a tuple, so a copy would be pointless.

But a tuple can *contain* a mutable list. `copy.deepcopy(mixed)` then has an interesting job: the tuple itself may be reused while the list inside is cloned to a fresh object. The inner ids differ, proving the goods were duplicated even though the shell was shared.

The lesson: immutability is shallow. A tuple protects only its own slots, not the objects those slots point to. A nested mutable inside an immutable container is still mutable.

In [3]:
# 🐣 TUPLE-OF-LISTS edge — immutable shell, mutable goods! 🐚🎁
mixed = ([1, 2, 3], "label")           # tuple (immutable) with list inside!
s = mixed[:]                            # slicing on tuple → same object re-use 🐚
print("tuple slice same object?", s is mixed, "← immutable shells ko copy hi nahi banate!")

d = copy.deepcopy(mixed)
print("deepcopy tuple is new?", d is mixed, "← lekin INNER list ka naya clone!")
print("inner list ids differ:", id(mixed[0]) != id(d[0]), "✅ deep ne goods duplicate ki")
# Expected OUTPUT:
# tuple slice same object? True ← immutable shells ko copy hi nahi banate!
# deepcopy tuple is new? True ← lekin INNER list ka naya clone!
# inner list ids differ: True ✅ deep ne goods duplicate ki

tuple slice same object? True ← immutable shells ko copy hi nahi banate!
deepcopy tuple is new? False ← lekin INNER list ka naya clone!
inner list ids differ: True ✅ deep ne goods duplicate ki


## 5. The JSON Round-Trip Hack

A quick way to deep-copy JSON-compatible data is `json.loads(json.dumps(obj))`. Serialising to a string and parsing it back produces entirely new objects, so mutations to the clone do not affect the original. For configurations made of dictionaries, lists, strings, numbers, booleans, and null, this works well and reads clearly.

It is a hack, not a general solution. It only understands JSON types, it is slower than `copy.deepcopy`, and it silently changes some types. The next cell exposes those traps.

In [4]:
# 🐔 JSON-roundtrip jugaad — poora deep copy ek line me? 🔄
import json
config = {"app": "ShopKart", "features": ["search", "cart"], "limits": {"retries": 3}}

clone = json.loads(json.dumps(config))
clone["features"].append("checkout")
print("config:", config["features"])
print("clone :", clone["features"])
print("independent:", id(config["features"]) != id(clone["features"]), "✅ jugaad works!")
# Expected OUTPUT:
# config: ['search', 'cart']
# clone : ['search', 'cart', 'checkout']
# independent: True ✅ jugaad works!

config: ['search', 'cart']
clone : ['search', 'cart', 'checkout']
independent: True ✅ jugaad works!


## 6. JSON Traps — Types That Bite

Three specific traps make the JSON round-trip risky.

1. A **set** is not JSON serialisable, so `json.dumps` raises `TypeError`. The set must be converted to a list first.
2. A **tuple** becomes a list after the round-trip, so `(1, 2)` returns as `[1, 2]`.
3. Non-string dictionary **keys** are converted to strings, so `{42: 'answer'}` becomes `{'42': 'answer'}`.

| Original type | After JSON round-trip |
|---|---|
| `set` | `TypeError` |
| `tuple` | `list` |
| int key `42` | string key `'42'` |

Use the JSON trick only when the data is plain JSON and you accept these conversions; otherwise prefer `copy.deepcopy`.

In [5]:
# ⚠️ JSON jugaad ke 3 TRAPS — types ka dhamaka! 💣
weird = {"tuple": (1, 2), "intkey": 42, "set": {1, 2}}
try:
    json.loads(json.dumps(weird))
except TypeError as e:
    print("set trap 🚨:", e)

# Tuple silently LIST ban jata, int keys STR hote 🧨
partial = json.loads(json.dumps({"tuple": (1, 2), 42: "answer"}))
print("tuple→", type(partial["tuple"]).__name__)
print("keys →", list(partial.keys()), "| types:", [type(k).__name__ for k in partial.keys()])
# Expected OUTPUT:
# set trap 🚨: Object of type set is not JSON serializable
# tuple→ list
# keys → ['tuple', '42'] | types: ['str', 'str']

set trap 🚨: Object of type set is not JSON serializable
tuple→ list
keys → ['tuple', '42'] | types: ['str', 'str']


## 7. Decision Flowchart

```text
Need a copy?
 |- Flat (numbers/strings only)? -> SHALLOW (fast: [:] or .copy())
 |- Nested lists/dicts inside?
 |    |- Only reading the copy? -> SHALLOW is fine (shared, read-only)
 |    -| Inner mutation possible? -> DEEPCOPY (copy.deepcopy)
 |- Immutable shell (tuple)? -> .copy() is a no-op; inner objects are still shared
 -| JSON-safe types and a quick fix? -> json round-trip (watch the type traps!)
```

Read the flowchart top to bottom. The decisive question is always whether **nested mutable data can change**; if it can, choose deep copy. This flowchart is a practical checklist you can apply at any copy site in real code.

## ⚠️ 5 Common Mistakes (Diagram Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Assuming "nested safe" on Dict `.copy()` | Outer-fresh only! Deep copy for the inner 💔 |
| 2 | Thinking a tuple slice "made a copy" | Copy of immutables = the same object (optimization!) |
| 3 | JSON hack without type-checks | set ❌, tuple→list 🧨, int-keys→str |
| 4 | Ignoring the deepcopy cost (big trees are slow!) | Deepcopy only the needed part; shallow on flat |
| 5 | Forgetting the `id()` check in trial runs | The id mirror → the truth! (build the diagram in your mind) |

> 🎯 **Compass-meter:** flat→shallow(✈️fast), nested-mutate→deep(🛡️safe),
> on-debug→id() experiment! — memory never lies 🪞

## Summary

- A memory diagram makes it clear: shallow copies rebuild the shelf, deep copies rebuild the goods.
- Nested dictionaries and lists are shared by a shallow copy, so edits leak back to the original.
- `copy.deepcopy` gives full independence for nested structures you intend to modify.
- Tuples are reused by slicing; only a deep copy rebuilds the mutable objects they contain.
- The JSON round-trip is a readable deep copy for JSON types, but sets fail and tuples and integer keys change type.
- Decide with one question: can nested mutable data change? If yes, deep copy.